# Apêndice 03b — Por que usamos `StandardScaler` e não `RobustScaler`O planejamento previa `RobustScaler`. Este apêndice mostra por quetrocamos. Ele não faz parte do pipeline; é só a justificativa da escolhafeita no notebook 03.O peso `1/√n` por bloco só funciona se cada coluna padronizada tivervariância 1. Se o escalonador não garante isso, o peso não deixa os blocosiguais. Comparamos três opções, todas com o mesmo `log1p` e o mesmo peso:| | Contínuas | Ordinais do IEGM ||---|---|---|| **A** | `RobustScaler` | `RobustScaler` (o plano original) || **B** | `RobustScaler` | escala 1–5, só centrada || **C** | `StandardScaler` | `StandardScaler` |

In [ ]:
import sysfrom pathlib import Pathimport numpy as npimport pandas as pdfrom sklearn.decomposition import PCAfrom sklearn.preprocessing import RobustScaler, StandardScalerRAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(RAIZ / "src"))from config import BASE_FINAL_CSV, DICIONARIO_CSVbase = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})dic = pd.read_csv(DICIONARIO_CSV)bloco_de = dic.set_index("coluna")["bloco"]ORDEM_BLOCOS = ["criminalidade", "socioeconomico", "gestao"]features = sorted(dic.loc[dic["papel"] == "feature", "coluna"],                  key=lambda c: (ORDEM_BLOCOS.index(bloco_de[c]), c))modelagem = base[~base["flag_sem_iegm"]].reset_index(drop=True)

In [ ]:
log_cols = ([c for c in features if bloco_de[c] == "criminalidade"]            + ["pib_percapita", "renda_domiciliar_mediana"])ordinais = [c for c in features if bloco_de[c] == "gestao"]continuas = [c for c in features if c not in ordinais]X = modelagem[features].copy()X[log_cols] = np.log1p(X[log_cols])n_bloco = pd.Series([bloco_de[c] for c in features]).value_counts()peso = pd.Series({c: 1 / np.sqrt(n_bloco[bloco_de[c]]) for c in features})

In [ ]:
def escalar(opcao):    Z = pd.DataFrame(index=X.index, columns=features, dtype=float)    if opcao == "A":        Z[features] = RobustScaler().fit_transform(X[features])    elif opcao == "B":        Z[continuas] = RobustScaler().fit_transform(X[continuas])        Z[ordinais] = (X[ordinais] - 1) / 4        # 1..5 vira 0..1        Z[ordinais] = Z[ordinais] - Z[ordinais].mean()    else:        Z[features] = StandardScaler().fit_transform(X[features])    return Zdef orcamento(M):    var = M.var(ddof=0)    return (var.groupby(M.columns.map(bloco_de)).sum()               .reindex(ORDEM_BLOCOS) / var.sum() * 100).round(1)

In [ ]:
linhas = {}for opcao in ["A", "B", "C"]:    W = escalar(opcao) * peso    pca = PCA().fit(W)    ev = pca.explained_variance_ratio_    carga_pc2 = pd.Series(pca.components_[1], index=features).abs()    linhas[opcao] = {        **orcamento(W).to_dict(),        "componentes p/ 80%": int(np.argmax(np.cumsum(ev) >= 0.80)) + 1,        "maior carga na PC2": f"{carga_pc2.idxmax()} ({carga_pc2.max():.2f})",    }pd.DataFrame(linhas).T

## O que a tabela mostra**A — `RobustScaler` em tudo.** Os blocos não ficam iguais (por volta de34 / 36 / 30), e a segunda componente principal é praticamente uma variávelsó: `i_planejamento_ord`, com carga acima de 0,9. O motivo é simples: o`RobustScaler` divide pelo intervalo interquartil (IQR), e o IQR de`i_planejamento_ord` é 0,33, porque a maioria dos municípios tem o mesmovalor. Dividir por 0,33 multiplica a coluna por três. Ou seja, oescalonador aumenta justamente a variável que tem menos informação.**B — escala 1–5 nas ordinais.** Manter a escala original sem reescalarderruba o bloco de gestão para uns 2% da distância. Como a gestão pública éo diferencial do trabalho, isso não serve.**C — `StandardScaler` em tudo.** Cada bloco fica com 33,3%, que é o que opeso `1/√n` promete, e nenhuma variável sozinha domina uma componente.## DecisãoUsamos `StandardScaler`. O motivo original para o `RobustScaler` erasegurar os valores extremos, mas isso o `log1p` já faz (no notebook 02, aassimetria depois do `log1p` fica entre −0,5 e +0,7 sem os zeros). Aplicar`RobustScaler` por cima não ajuda e ainda quebra a igualdade entre blocos.

In [ ]:
iqr = (X[ordinais].quantile(0.75) - X[ordinais].quantile(0.25)).round(3)iqr.rename("IQR (divisor do RobustScaler)").to_frame()